<a href="https://colab.research.google.com/github/moath177/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moath177/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

##Unit of analysis:
 one row = one content item `(content_hash_id)`, aggregated over a single month window.

Tables: `dim_content` (content metadata) joined to `fact_content_daily_performance` (daily search performance, filtered to one month partition).

##Time window:
development and verification are done on `month = 2026-03` — a mid-panel month, not the final month. The `_sample` table (June 2026) is explicitly avoided for anything beyond mechanics testing, since it is the natural outcome window for any past→future label and would leak into later modeling weeks.

In [2]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

Paste your Hugging Face READ token (hf_...): ··········


In [5]:
grain_check = con.sql(f"""
    SELECT content_hash_id, COUNT(DISTINCT client_hash_id) AS n_clients
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id
    HAVING n_clients > 1
    LIMIT 5
""").df()

len(grain_check)

0

In [7]:
window_check = con.sql(f"""
    SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date, COUNT(*) AS n_rows
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
len(window_check)

1

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

**Field classification for Lane 4 (CTR/Engagement Opportunity Scoring), month = 2026-03:**

### Context
*(grouping, joining, or filtering — never fed to the model)*
- `content_hash_id`, `client_hash_id` — join/group keys only (verified: no content item spans multiple clients in this month)
- `report_date` / `month` — time window key
- `is_published`, `is_deleted` — used only to define the analysis slice: **visible = `is_published IS TRUE AND is_deleted IS FALSE`**. Not passed to the model.

### Label / proxy
- `ctr = gsc_clicks / gsc_impressions` — raw click-through rate for this first warehouse pass. Not position/tier-adjusted yet (opportunity_gap deferred to a later iteration).

### Feature
*(knowable before the decision moment, safe to use)*
- `gsc_avg_position`
- `gsc_impressions`
- `content_type`
- `word_count`
- content age (derived: `report_date - content_created_date`)

### Excluded
- `gsc_clicks` — used only to compute the label; excluded as a feature to avoid direct leakage (the label is literally built from it).
- GA4/engagement columns (e.g. sessions, engagement_rate) — not used in this pass. Would require filtering on `ga4_data_available IS TRUE` first, since rows before a client's GA4 start are zero-filled, not truly zero-engagement. Deferred, not forgotten.

gsc_clicks — used only to compute the label; excluded as a feature to avoid direct leakage (the label is literally built from it).
GA4/engagement columns (e.g. sessions, engagement_rate) — not used in this pass. Would require filtering on ga4_data_available IS TRUE first, since rows before a client's GA4 start are zero-filled, not truly zero-engagement. Deferred, not forgotten.

In [9]:
schema_check = con.sql(f"""
    SELECT column_name, column_type
    FROM (DESCRIBE SELECT * FROM read_parquet('{REL}/dim_content.parquet'))
    WHERE column_name IN ('content_hash_id','content_type','word_count','content_created_date','is_published','is_deleted')

    UNION ALL

    SELECT column_name, column_type
    FROM (DESCRIBE SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet'))
    WHERE column_name IN ('client_hash_id','content_hash_id','report_date','gsc_avg_position','gsc_impressions','gsc_clicks')
""").df()

schema_check

,column_name,column_type
0,content_hash_id,VARCHAR
1,content_created_date,DATE
2,content_type,VARCHAR
3,word_count,BIGINT
4,is_published,BOOLEAN
5,is_deleted,BOOLEAN
6,report_date,DATE
7,client_hash_id,VARCHAR
8,content_hash_id,VARCHAR
9,gsc_impressions,BIGINT


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS n_rows_visible,
        COUNT(DISTINCT f.content_hash_id) AS n_content_items,
        MIN(f.report_date) AS min_date,
        MAX(f.report_date) AS max_date
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') f
    JOIN read_parquet('{REL}/dim_content.parquet') d
        ON f.content_hash_id = d.content_hash_id
    WHERE d.is_published IS TRUE
      AND d.is_deleted IS FALSE
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows_visible,n_content_items,min_date,max_date
0,9532718,321106,2026-03-01,2026-03-31


In [11]:
total_check = con.sql(f"""
    SELECT COUNT(*) AS n_rows_total
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

total_check

,n_rows_total
0,9841378


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.